# Log-Odds Evidence Applet

A compact teaching applet for clinical learners. The central idea: detective mode uses log-odds because evidence adds; action mode uses probability because decisions need expected consequences.


## Learning Goal

Use the controls below to see why likelihood ratios are awkward on a probability ruler but simple on a log-odds ruler. Strong evidence moves odds by multiplication, which becomes addition after taking logs.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import HTML, display

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

EPS = 1e-6
RULER_PROBS = [0.01, 0.05, 0.10, 0.20, 0.50, 0.80, 0.90, 0.95, 0.99]
RULER_XMIN = math.log(0.01 / 0.99)
RULER_XMAX = math.log(0.99 / 0.01)

BLUE = "#1f77b4"
GREEN = "#2ca02c"
RED = "#d62728"
ORANGE = "#ff7f0e"
PURPLE = "#9467bd"
GRAY = "#6b7280"
LIGHT_GRAY = "#e5e7eb"


def clamp_probability(probability):
    return min(max(float(probability), EPS), 1 - EPS)


def probability_to_odds(probability):
    p = clamp_probability(probability)
    return p / (1 - p)


def odds_to_probability(odds):
    odds = max(float(odds), 0)
    return odds / (1 + odds)


def prob_to_logodds(probability):
    return math.log(probability_to_odds(probability))


def logodds_to_prob(logodds):
    if logodds >= 0:
        z = math.exp(-logodds)
        return 1 / (1 + z)
    z = math.exp(logodds)
    return z / (1 + z)


def bayes_update_probability(prior_probability, likelihood_ratio):
    return odds_to_probability(probability_to_odds(prior_probability) * float(likelihood_ratio))


def lr_for_probability_move(before_probability, after_probability):
    return probability_to_odds(after_probability) / probability_to_odds(before_probability)


def pct(probability, digits=1):
    return f"{100 * probability:.{digits}f}%"


def fmt_num(value, digits=3):
    if abs(value) >= 10:
        return f"{value:.1f}"
    return f"{value:.{digits}f}"


def fmt_lr(likelihood_ratio):
    return fmt_num(likelihood_ratio, 3)


def decibans_from_lr(likelihood_ratio):
    return 10 * math.log10(likelihood_ratio)


def html_box(title, body):
    return HTML(
        f"""
        <div style="border-left: 4px solid #1f77b4; padding: 0.6rem 0.8rem; margin: 0.4rem 0 0.7rem 0; background: #f8fafc;">
          <div style="font-weight: 700; margin-bottom: 0.2rem;">{title}</div>
          <div>{body}</div>
        </div>
        """
    )


def metric_table(rows):
    row_html = "".join(
        f"<tr><th style='text-align:left; padding: 0.2rem 0.7rem 0.2rem 0;'>{label}</th>"
        f"<td style='text-align:right; padding: 0.2rem 0;'>{value}</td></tr>"
        for label, value in rows
    )
    return HTML(f"<table style='border-collapse: collapse; margin: 0.3rem 0;'>{row_html}</table>")


In [ ]:
def _decorate_axis_as_ruler(ax, xlim, title, xlabel, ticks, labels):
    ax.set_xlim(*xlim)
    ax.set_ylim(-0.55, 0.55)
    ax.axhline(0, color="#111827", linewidth=1.2)
    ax.set_yticks([])
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels)
    ax.set_title(title, loc="left", pad=8)
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", linestyle=":", linewidth=0.6, alpha=0.65)


def _draw_arrow(ax, start, end, color, y=0.0, label=None):
    ax.scatter([start, end], [y, y], s=52, color=[color, color], zorder=3)
    ax.annotate(
        "",
        xy=(end, y),
        xytext=(start, y),
        arrowprops=dict(arrowstyle="->", color=color, lw=2.4, shrinkA=4, shrinkB=4),
    )
    if label:
        midpoint = (start + end) / 2
        ax.text(midpoint, y + 0.17, label, ha="center", va="bottom", color=color, fontsize=9)


def make_update_rulers(prior, likelihood_ratio):
    post = bayes_update_probability(prior, likelihood_ratio)
    prior_logodds = prob_to_logodds(prior)
    post_logodds = prior_logodds + math.log(likelihood_ratio)
    log_ticks = [prob_to_logodds(p) for p in RULER_PROBS]
    prob_labels = [pct(p, 0) if p not in (0.01, 0.99) else pct(p, 0) for p in RULER_PROBS]

    fig, axes = plt.subplots(2, 1, figsize=(8.4, 3.9), constrained_layout=True)
    _decorate_axis_as_ruler(
        axes[0],
        (0, 1),
        "Probability ruler: the same LR does not have the same length everywhere",
        "Probability",
        RULER_PROBS,
        prob_labels,
    )
    _draw_arrow(axes[0], prior, post, BLUE, label=f"{pct(prior)} -> {pct(post)}")

    _decorate_axis_as_ruler(
        axes[1],
        (RULER_XMIN, RULER_XMAX),
        "Log-odds ruler: the same LR always has length log(LR)",
        "Probability labels placed at equal log-odds positions",
        log_ticks,
        prob_labels,
    )
    _draw_arrow(axes[1], prior_logodds, post_logodds, GREEN, label=f"log(LR) = {math.log(likelihood_ratio):.3f}")
    return fig, post


def make_move_requirement_plot(before, after):
    log_before = prob_to_logodds(before)
    log_after = prob_to_logodds(after)
    log_ticks = [prob_to_logodds(p) for p in RULER_PROBS]
    prob_labels = [pct(p, 0) for p in RULER_PROBS]

    fig, axes = plt.subplots(2, 1, figsize=(8.4, 3.7), constrained_layout=True)
    _decorate_axis_as_ruler(
        axes[0],
        (0, 1),
        "Requested probability move",
        "Probability",
        RULER_PROBS,
        prob_labels,
    )
    _draw_arrow(axes[0], before, after, BLUE, label=f"{pct(before)} -> {pct(after)}")

    _decorate_axis_as_ruler(
        axes[1],
        (RULER_XMIN, RULER_XMAX),
        "Evidence required for that move",
        "Log-odds scale",
        log_ticks,
        prob_labels,
    )
    _draw_arrow(axes[1], log_before, log_after, ORANGE, label=f"required log(LR) = {log_after - log_before:.3f}")
    return fig


def make_logodds_path_plot(prior, evidence_steps):
    prior_logodds = prob_to_logodds(prior)
    cumulative = 0
    xs = [prior_logodds]
    for _, weight, _ in evidence_steps:
        cumulative += weight
        xs.append(prior_logodds + cumulative)

    left = min(RULER_XMIN, min(xs) - 0.4)
    right = max(RULER_XMAX, max(xs) + 0.4)
    log_ticks = [prob_to_logodds(p) for p in RULER_PROBS]
    prob_labels = [pct(p, 0) for p in RULER_PROBS]

    fig, ax = plt.subplots(figsize=(8.4, 2.4), constrained_layout=True)
    _decorate_axis_as_ruler(ax, (left, right), "Cumulative evidence on the log-odds ruler", "Probability labels", log_ticks, prob_labels)

    current = prior_logodds
    ax.scatter([current], [0], s=60, color=BLUE, zorder=3)
    ax.text(current, -0.24, "prior", ha="center", va="top", fontsize=8, color=BLUE)
    if not evidence_steps:
        ax.text(current, 0.17, "add evidence chips", ha="center", va="bottom", fontsize=9, color=GRAY)
        return fig

    y_offsets = np.linspace(0.16, -0.16, max(len(evidence_steps), 1))
    for (label, weight, color), y in zip(evidence_steps, y_offsets):
        nxt = current + weight
        _draw_arrow(ax, current, nxt, color, y=y, label=label)
        current = nxt
    ax.scatter([current], [0], s=70, color=GREEN, zorder=4)
    ax.text(current, -0.24, "after", ha="center", va="top", fontsize=8, color=GREEN)
    return fig


BIN_BOUNDARIES_LOGODDS = [-1.5, -0.5, 0.5, 1.5]
BIN_BOUNDARIES_PROB = [logodds_to_prob(x) for x in BIN_BOUNDARIES_LOGODDS]
BIN_LABELS = ["Highly unlikely", "Unlikely", "Uncertain", "Likely", "Highly likely"]
BIN_REP_LOGODDS = [-2, -1, 0, 1, 2]
BIN_REP_PROBS = [logodds_to_prob(x) for x in BIN_REP_LOGODDS]
BIN_COLORS = ["#f3f4f6", "#dbeafe", "#fef3c7", "#dcfce7", "#ede9fe"]

FEATURE_TYPES = [
    ("Major +", 1.0, GREEN, "LR 2.718"),
    ("Minor +", 1.0 / 3.0, BLUE, "LR 1.396"),
    ("Minor -", -1.0 / 6.0, ORANGE, "LR 0.846"),
    ("Major -", -0.5, RED, "LR 0.607"),
]


def make_bin_plot(start_logodds, cumulative_weight, show_thresholds=True):
    end_logodds = start_logodds + cumulative_weight
    left = min(-3.2, start_logodds, end_logodds) - 0.2
    right = max(3.2, start_logodds, end_logodds) + 0.2
    fig, ax = plt.subplots(figsize=(8.4, 2.5), constrained_layout=True)

    edges = [left, *BIN_BOUNDARIES_LOGODDS, right]
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        color = BIN_COLORS[min(i, len(BIN_COLORS) - 1)]
        ax.axvspan(lo, hi, color=color, alpha=0.95, zorder=0)
        label_x = (lo + hi) / 2
        if left <= label_x <= right:
            ax.text(label_x, 0.37, BIN_LABELS[min(i, len(BIN_LABELS) - 1)], ha="center", va="center", fontsize=8)

    if show_thresholds:
        for boundary, p in zip(BIN_BOUNDARIES_LOGODDS, BIN_BOUNDARIES_PROB):
            ax.axvline(boundary, color=GRAY, linestyle="--", linewidth=0.8)
            ax.text(boundary, -0.38, pct(p), ha="center", va="top", fontsize=8, color=GRAY)

    _decorate_axis_as_ruler(
        ax,
        (left, right),
        "Qualitative categories are evenly spaced in log-odds",
        "Log-odds, with probability thresholds shown when enabled",
        BIN_BOUNDARIES_LOGODDS,
        [pct(p) for p in BIN_BOUNDARIES_PROB],
    )
    _draw_arrow(ax, start_logodds, end_logodds, PURPLE, label=f"total WoE = {cumulative_weight:.3f}")
    return fig


def category_for_probability(probability):
    logodds = prob_to_logodds(probability)
    index = sum(logodds >= b for b in BIN_BOUNDARIES_LOGODDS)
    return BIN_LABELS[index]


In [ ]:
def _verification_smoke_tests():
    assert math.isclose(bayes_update_probability(0.5, 2), 2 / 3, rel_tol=1e-6)
    assert math.isclose(lr_for_probability_move(0.7, 0.8), 1.7142857142857146, rel_tol=1e-6)
    assert math.isclose(lr_for_probability_move(0.8, 0.9), 2.25, rel_tol=1e-6)
    assert math.isclose(lr_for_probability_move(0.05, 0.15), 3.3529411764705883, rel_tol=1e-6)
    assert math.isclose(math.exp(-0.5), 0.6065306597126334, rel_tol=1e-6)
    assert math.isclose(math.exp(-1 / 6), 0.8464817248906141, rel_tol=1e-6)
    step = math.log(2)
    assert math.isclose((prob_to_logodds(0.5) + step) - prob_to_logodds(0.5), step, rel_tol=1e-12)


_verification_smoke_tests()


In [ ]:
def panel_same_evidence():
    prior = widgets.FloatSlider(value=0.50, min=0.01, max=0.99, step=0.01, description="Prior", readout_format=".2f", continuous_update=False)
    lr = widgets.FloatLogSlider(value=2.0, base=10, min=-1, max=1, step=0.01, description="LR", readout_format=".2f", continuous_update=False)
    out = widgets.Output()

    def update(_=None):
        with out:
            out.clear_output(wait=True)
            fig, post = make_update_rulers(prior.value, lr.value)
            display(fig)
            plt.close(fig)
            comparison_priors = [0.05, 0.50, 0.90]
            comparison = []
            for p in comparison_priors:
                after = bayes_update_probability(p, lr.value)
                comparison.append((f"{pct(p)} prior", f"{pct(after)} after; change {100 * (after - p):+.1f} points"))
            display(metric_table([
                ("Selected prior", pct(prior.value)),
                ("Likelihood ratio", fmt_lr(lr.value)),
                ("Post-test probability", pct(post)),
                ("Weight of evidence", f"log(LR) = {math.log(lr.value):.3f}"),
            ]))
            display(html_box("Same evidence, different probability move", "A fixed LR has fixed length on the log-odds ruler. Its probability-point movement depends on where you start."))
            display(metric_table(comparison))

    prior.observe(update, names="value")
    lr.observe(update, names="value")
    update()

    return widgets.VBox([
        widgets.HTML("<h3>1. Same Evidence, Different Probability Move</h3>"),
        widgets.HTML("<p>Move the prior and LR. The lower ruler is the natural evidence scale.</p>"),
        widgets.HBox([prior, lr]),
        out,
    ])


In [ ]:
def panel_required_evidence():
    before = widgets.FloatSlider(value=0.05, min=0.01, max=0.99, step=0.01, description="Before", readout_format=".2f", continuous_update=False)
    after = widgets.FloatSlider(value=0.15, min=0.01, max=0.99, step=0.01, description="After", readout_format=".2f", continuous_update=False)
    out = widgets.Output()

    def update(_=None):
        with out:
            out.clear_output(wait=True)
            if math.isclose(before.value, after.value):
                required_lr = 1.0
            else:
                required_lr = lr_for_probability_move(before.value, after.value)
            required_loglr = math.log(required_lr)
            fig = make_move_requirement_plot(before.value, after.value)
            display(fig)
            plt.close(fig)

            contrast_rows = []
            for start, finish in [(0.05, 0.15), (0.50, 0.60), (0.80, 0.90), (0.70, 0.80)]:
                lr_needed = lr_for_probability_move(start, finish)
                contrast_rows.append((f"{pct(start, 0)} -> {pct(finish, 0)}", f"LR {fmt_lr(lr_needed)}; log(LR) {math.log(lr_needed):.3f}"))

            display(metric_table([
                ("Required LR", fmt_lr(required_lr)),
                ("Required log(LR), nats", f"{required_loglr:.3f}"),
                ("Decibans", f"{decibans_from_lr(required_lr):.2f}"),
            ]))
            display(html_box("Key contrast", "A 5% -> 15% move needs LR 3.353. A 50% -> 60% move needs LR 1.500, even though both are 10 probability points."))
            display(metric_table(contrast_rows))

    before.observe(update, names="value")
    after.observe(update, names="value")
    update()

    return widgets.VBox([
        widgets.HTML("<h3>2. How Much Evidence Did That Move Require?</h3>"),
        widgets.HTML("<p>Choose two probabilities. The app computes the LR that would be needed to get from before to after.</p>"),
        widgets.HBox([before, after]),
        out,
    ])


In [ ]:
def _chip_html(evidence_steps):
    if not evidence_steps:
        return "<span style='color:#6b7280;'>No evidence chips added yet.</span>"
    chips = []
    for label, weight, color in evidence_steps:
        chips.append(
            f"<span style='display:inline-block; padding:0.18rem 0.45rem; margin:0.1rem; "
            f"border-radius:0.35rem; border:1px solid {color}; color:{color}; background:#ffffff;'>"
            f"{label} ({weight:+.3f})</span>"
        )
    return "".join(chips)


def panel_evidence_adds():
    prior = widgets.FloatSlider(value=0.50, min=0.01, max=0.99, step=0.01, description="Prior", readout_format=".2f", continuous_update=False)
    out = widgets.Output()
    evidence_steps = []

    buttons = []
    for label, weight, color, _ in FEATURE_TYPES:
        button = widgets.Button(description=label, layout=widgets.Layout(width="7rem"))
        button.style.button_color = "#ffffff"

        def on_click(_button, label=label, weight=weight, color=color):
            evidence_steps.append((label, weight, color))
            update()

        button.on_click(on_click)
        buttons.append(button)

    reset = widgets.Button(description="Reset", icon="refresh", layout=widgets.Layout(width="7rem"))

    def reset_click(_button):
        evidence_steps.clear()
        update()

    reset.on_click(reset_click)

    def update(_=None):
        with out:
            out.clear_output(wait=True)
            cumulative = sum(weight for _, weight, _ in evidence_steps)
            prior_logodds = prob_to_logodds(prior.value)
            post = logodds_to_prob(prior_logodds + cumulative)
            fig = make_logodds_path_plot(prior.value, evidence_steps)
            display(fig)
            plt.close(fig)
            display(HTML(f"<div style='margin:0.4rem 0;'>{_chip_html(evidence_steps)}</div>"))
            display(metric_table([
                ("Prior log-odds", f"{prior_logodds:.3f}"),
                ("Sum of evidence", f"{cumulative:+.3f}"),
                ("After log-odds", f"{prior_logodds + cumulative:.3f}"),
                ("After probability", pct(post)),
                ("Cumulative LR", fmt_lr(math.exp(cumulative))),
            ]))
            display(html_box("Evidence adds", "Each chip changes log-odds by a fixed amount. Multiplication of LRs becomes addition of weights of evidence."))

    prior.observe(update, names="value")
    update()

    return widgets.VBox([
        widgets.HTML("<h3>3. Evidence Adds</h3>"),
        widgets.HTML("<p>Add positive or negative findings. Watch the total move linearly before it is converted back to probability.</p>"),
        prior,
        widgets.HBox([*buttons, reset]),
        out,
    ])


In [ ]:
def panel_qualitative_bins():
    category_options = [(f"{label} ({pct(prob)})", idx) for idx, (label, prob) in enumerate(zip(BIN_LABELS, BIN_REP_PROBS))]
    start_category = widgets.Dropdown(options=category_options, value=2, description="Start")
    major_pos = widgets.IntSlider(value=0, min=0, max=5, step=1, description="Major +", continuous_update=False)
    minor_pos = widgets.IntSlider(value=0, min=0, max=6, step=1, description="Minor +", continuous_update=False)
    minor_neg = widgets.IntSlider(value=0, min=0, max=6, step=1, description="Minor -", continuous_update=False)
    major_neg = widgets.IntSlider(value=0, min=0, max=5, step=1, description="Major -", continuous_update=False)
    show_thresholds = widgets.Checkbox(value=True, description="Show thresholds")
    out = widgets.Output()

    def update(_=None):
        start_logodds = BIN_REP_LOGODDS[start_category.value]
        start_prob = logodds_to_prob(start_logodds)
        cumulative = (
            major_pos.value * 1.0
            + minor_pos.value * (1.0 / 3.0)
            - minor_neg.value * (1.0 / 6.0)
            - major_neg.value * 0.5
        )
        after_logodds = start_logodds + cumulative
        after_prob = logodds_to_prob(after_logodds)

        with out:
            out.clear_output(wait=True)
            fig = make_bin_plot(start_logodds, cumulative, show_thresholds.value)
            display(fig)
            plt.close(fig)
            display(metric_table([
                ("Start", f"{BIN_LABELS[start_category.value]} ({pct(start_prob)})"),
                ("Total weight of evidence", f"{cumulative:+.3f}"),
                ("Total LR", fmt_lr(math.exp(cumulative))),
                ("After", f"{category_for_probability(after_prob)} ({pct(after_prob)})"),
            ]))
            display(metric_table([
                ("Major positive", "+1.000 WoE; LR 2.718"),
                ("Minor positive", "+0.333 WoE; LR 1.396"),
                ("Minor negative", "-0.167 WoE; LR 0.846"),
                ("Major negative", "-0.500 WoE; LR 0.607"),
            ]))
            display(html_box("Qualitative Bayes bins", "These bins are not evenly spaced in probability. They are evenly spaced where evidence adds: log-odds."))

    for widget in [start_category, major_pos, minor_pos, minor_neg, major_neg, show_thresholds]:
        widget.observe(update, names="value")
    update()

    controls = widgets.VBox([
        start_category,
        widgets.HBox([major_pos, minor_pos]),
        widgets.HBox([minor_neg, major_neg]),
        show_thresholds,
    ])
    return widgets.VBox([
        widgets.HTML("<h3>4. Qualitative Bayes Bins</h3>"),
        widgets.HTML("<p>Major and minor findings are rough evidence units. The corrected negative LRs are shown explicitly.</p>"),
        controls,
        out,
    ])


In [ ]:
tabs = widgets.Tab(children=[
    panel_same_evidence(),
    panel_required_evidence(),
    panel_evidence_adds(),
    panel_qualitative_bins(),
])
for index, title in enumerate(["Same evidence", "Required evidence", "Evidence adds", "Qualitative bins"]):
    tabs.set_title(index, title)

app = widgets.VBox([
    widgets.HTML("<h2>Log-Odds Evidence Ruler</h2>"),
    widgets.HTML("<p><b>Detective mode</b> uses log-odds because evidence adds. <b>Action mode</b> uses probability because decisions need expected consequences.</p>"),
    tabs,
    widgets.HTML("<p style='margin-top:1rem; color:#4b5563;'><b>Next:</b> in a multi-diagnosis differential, evidence for one diagnosis necessarily redistributes probability away from alternatives.</p>"),
])

display(app)


## Source Anchors

This applet distills the log-odds and qualitative Bayes portions of `display_reasoning.ipynb`. Later versions can add clinical differential diagnosis and trial re-evaluation modules.
